# CatBoost Lossguide (Asymmetric Trees)

CatBoost default uses **oblivious (symmetric) trees** — every node at a given depth
uses the same split. Fast and regularized, but less expressive.

`grow_policy='Lossguide'` builds **asymmetric trees** like XGBoost/LightGBM —
each leaf is split independently based on loss reduction. Completely different
split structure → diverse predictions for ensembling.

**Baselines**:
- CatBoost default (oblivious): CV AUC = 0.95533, LB = 0.95356

In [1]:
import numpy as np
import pandas as pd
import subprocess
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import catboost as cb

KAGGLE_DATA = Path('/kaggle/input/playground-series-s6e2')
LOCAL_DATA  = Path('data')
DATA_DIR    = KAGGLE_DATA if KAGGLE_DATA.exists() else LOCAL_DATA

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    if 'heart_disease' in df.columns:
        df['heart_disease'] = df['heart_disease'].map({'Absence': 0, 'Presence': 1})
    return df

train = prep(pd.read_csv(DATA_DIR / 'train.csv'))
test  = prep(pd.read_csv(DATA_DIR / 'test.csv'))
ss    = pd.read_csv(DATA_DIR / 'sample_submission.csv')

FEATURES     = [c for c in train.columns if c not in ['heart_disease', 'id']]
CAT_FEATURES = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
                'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']

X      = train[FEATURES]
y      = train['heart_disease'].values
X_test = test[FEATURES]

BASELINE_CV = 0.95533
BASELINE_LB = 0.95356
print(f'Train: {X.shape}  Test: {X_test.shape}')

Train: (630000, 13)  Test: (270000, 13)


## 5-Fold CV — Lossguide vs Oblivious

Lossguide requires `max_leaves` instead of `depth`. Equivalent expressiveness
to depth=6 oblivious (2^6=64 leaves) is `max_leaves=64`, but Lossguide trees
tend to be shallower and wider — try 31 and 64.

In [2]:
configs = [
    {'grow_policy': 'SymmetricTree', 'depth': 6,  'label': 'oblivious_d6   (baseline)'},
    {'grow_policy': 'Lossguide',     'max_leaves': 31,  'min_data_in_leaf': 1,  'label': 'lossguide_l31'},
    {'grow_policy': 'Lossguide',     'max_leaves': 64,  'min_data_in_leaf': 1,  'label': 'lossguide_l64'},
]

cv5     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for cfg in configs:
    label = cfg.pop('label')
    oof        = np.zeros(len(y))
    test_folds = np.zeros((5, len(X_test)))

    for fold, (tr_idx, val_idx) in enumerate(cv5.split(X, y)):
        m = cb.CatBoostClassifier(
            iterations=2000,
            learning_rate=0.05,
            task_type='GPU',
            cat_features=CAT_FEATURES,
            random_state=42,
            verbose=0,
            eval_metric='AUC',
            early_stopping_rounds=50,
            **cfg
        )
        m.fit(
            X.iloc[tr_idx], y[tr_idx],
            eval_set=(X.iloc[val_idx], y[val_idx]),
            verbose=False
        )
        oof[val_idx]     = m.predict_proba(X.iloc[val_idx])[:, 1]
        test_folds[fold] = m.predict_proba(X_test)[:, 1]

    cv_auc = roc_auc_score(y, oof)
    results[label] = {'cv_auc': cv_auc, 'oof': oof,
                      'test_preds': test_folds.mean(axis=0)}
    print(f'{label}:  CV AUC = {cv_auc:.5f}  ({cv_auc - BASELINE_CV:+.5f})')

print(f'\nBaseline (oblivious d6, lr=0.1): {BASELINE_CV:.5f}')

Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


oblivious_d6   (baseline):  CV AUC = 0.95537  (+0.00004)


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


lossguide_l31:  CV AUC = 0.95533  (-0.00000)


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


lossguide_l64:  CV AUC = 0.95524  (-0.00009)

Baseline (oblivious d6, lr=0.1): 0.95533


## Submit Best Lossguide Config

In [3]:
# Find best lossguide result (exclude baseline)
lossguide_results = {k: v for k, v in results.items() if 'lossguide' in k}
best_label = max(lossguide_results, key=lambda k: lossguide_results[k]['cv_auc'])
best       = lossguide_results[best_label]

print(f'Best lossguide: {best_label}  CV AUC = {best["cv_auc"]:.5f}')

fname = f'submissions/cat_{best_label.strip()}.csv'
desc  = f'cat_{best_label.strip()} | cv_auc={best["cv_auc"]:.4f}'

sub = ss.copy()
sub['Heart Disease'] = best['test_preds']
sub.to_csv(fname, index=False)
print(f'Saved: {fname}')

r = subprocess.run(
    ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
     '-f', fname, '-m', desc],
    capture_output=True, text=True
)
status = 'submitted' if 'successfully' in r.stdout.lower() else r.stdout.strip()[:120]
print(f'Status: {status}')
print(f'Compare to baseline LB = {BASELINE_LB}')

Best lossguide: lossguide_l31  CV AUC = 0.95533
Saved: submissions/cat_lossguide_l31.csv
Status: submitted
Compare to baseline LB = 0.95356


## Ensemble: Oblivious + Lossguide

Even if Lossguide CV ≈ oblivious CV, their predictions are structurally different.
Simple average of the two may outperform either alone.

In [4]:
from scipy.optimize import minimize

# Load oblivious CatBoost OOF
oof_cat_path = Path('submissions/oof_cat.npy')
if oof_cat_path.exists():
    oof_obliv  = np.load(oof_cat_path)
    test_obliv = np.load('submissions/test_cat.npy')

    oof_lg     = best['oof']
    test_lg    = best['test_preds']

    # SLSQP optimised blend
    def neg_auc(w):
        return -roc_auc_score(y, w[0]*oof_obliv + w[1]*oof_lg)

    res = minimize(neg_auc, x0=[0.5, 0.5], method='SLSQP',
                   bounds=[(0,1),(0,1)],
                   constraints={'type':'eq','fun': lambda w: w.sum()-1})
    opt_auc = -res.fun
    w_obliv, w_lg = res.x

    print(f'Oblivious weight: {w_obliv:.3f}  Lossguide weight: {w_lg:.3f}')
    print(f'Blend OOF AUC:  {opt_auc:.5f}  ({opt_auc - BASELINE_CV:+.5f} vs baseline)')

    if opt_auc > BASELINE_CV:
        blend_preds = w_obliv * test_obliv + w_lg * test_lg
        fname2 = 'submissions/cat_obliv_lossguide_blend.csv'
        desc2  = f'cat_obliv_lossguide_blend | cv_auc={opt_auc:.4f}'
        sub2   = ss.copy()
        sub2['Heart Disease'] = blend_preds
        sub2.to_csv(fname2, index=False)
        r2 = subprocess.run(
            ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
             '-f', fname2, '-m', desc2],
            capture_output=True, text=True
        )
        print(f'Blend: {"submitted" if "successfully" in r2.stdout.lower() else r2.stdout[:80]}')
    else:
        print('Blend does not beat baseline — skipping submission.')
else:
    print('oof_cat.npy not found — run s6e2-gradient-boosting.ipynb first.')

Oblivious weight: 0.500  Lossguide weight: 0.500
Blend OOF AUC:  0.95539  (+0.00006 vs baseline)
Blend: submitted
